# Rapport final : Python pour la data-science 2A S1

Manon Bourguignat, Leila Héry, Franck Tran  
Chargé de TD : Lino Galiana

On pensera à lire le fichier README.md du repo qui contient notamment les informations sur les dépendances et les données.

## I. Contexte

### 1. Présentation du sujet

La pandémie de Covid-19 a ravivé et intensifié une méfiance marquée envers les discours officiels au sein des démocraties européennes. Loin de constituer un simple manque d'information individuel, l'adhésion aux thèses complotistes reflète souvent une rupture démocratique et épistémique plus profonde vis-à-vis des institutions établies.

À partir des données de l'European Social Survey (ESS Round 10, 2020), notre étude vise à dépasser les seuls déterminants socio-économiques traditionnels (diplôme, revenu). Nous distinguons notamment la défiance envers le système politique de celle dirigée contre le consensus scientifique, tout en mesurant l'effet de la radicalité idéologique (éloignement du centre sur l'axe gauche-droite) et de l'ancrage territorial.

Notre travail s'articule ainsi autour de la problématique suivante :

> **« Dans quelle mesure l’adhésion aux théories du complot en Europe traduit-elle une défiance institutionnelle et scientifique plutôt qu’un clivage socio-économique traditionnel ? »**


### 2. Présentation de la base de donnée utilisée

L'European Social Service (ESS) est une enquête menée régulièrement dans plusieurs pays européens pour étudier les opinions, les valeurs et les comportements des populations. Chaque round (correspondant à une enquête réalisée une année donnée) comprend des variables communes à l'ensemble des rounds, ainsi que des variables propres aux thèmes étudiés.

Dans ce projet, on utilisera les données de l’ESS Round 10 (noté égament ESS10), édition 3.3, collectées en 2020 dont les thèmes sont la démocratie et les contacts sociaux numériques.

Tous les détails (références, licenses, accès) concernant les données se situent dans le fichier README.md.

## II. Préparation des données

Cette partie consiste en la récupération, l'exploration et le nettoyage des données. Elle aboutit à un dataframe propre nommé df_ess_final qui sera utilisé dans la partie III.

De nombreuses fonctions et variables utilisées sont regroupées dans des scripts python, dans le dossier "scripts" qu'on importe ici :

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# On importe des fonctions définies dans preparation_data.py
from scripts.preparation_data import recuperer_donnees, select_colonne, suppr_colonne, renommer_colonne 

# On importe aussi les variables
from scripts.preparation_data import dico_doi, colonnes_a_garder, nouveaux_noms

### 1. Récupération des données

Le choix a été de récupérer les données de l'enquête de 2020 avec l'API dédiée pour deux raisons :
- Faciliter la reproductibilité : l'API est plus pratique que de chercher à récupérer les données sur le site
- L'infrastructure de recherche qui gère l'ESS recommande dans ses conditions d'utilisation (https://europeansocialsurvey.org/contact/disclaimer) de ne pas mettre les jeux de données de l’ESS à disposition sur des sites web externes.

Lorsqu’on exécute la fonction *recuperer_donnes* définie dans preparation_data.py, **il faut renseigner un "ESS User ID"**.

<details style="border: 1px solid #888; border-radius: 6px; padding: 8px;">
<summary style="font-weight: bold; cursor: pointer;">Où trouver mon ESS User ID ?</summary>

 Elle est disponible sur la page de l’API de l’ESS (https://ess.sikt.no/en/api) une fois connecté à un compte.
 
 On pourra également utiliser l'ESS User ID de Franck TRAN qu'il vous aura transmis par une autre voie. 
 
 Cet identifiant ne sert pas à s’authentifier, mais à suivre les statistiques d’utilisation des données. 

In [6]:
df_ess_2020 = recuperer_donnees(2020)
print(df_ess_2020.shape)
print(df_ess_2020.columns)

(37611, 621)
Index(['name', 'essround', 'edition', 'proddate', 'idno', 'cntry', 'dweight',
       'pspwght', 'pweight', 'anweight',
       ...
       'vinwe', 'inwde', 'jinws', 'jinwe', 'inwtm', 'mode', 'domain', 'prob',
       'stratum', 'psu'],
      dtype='str', length=621)


On observe qu'il y a +37000 échantillons et +621 variables avec des noms pas très explicites comme 'idno' pour désigner le numéro d’identification du répondant.

On conserve uniquement les colonnes susceptibles de contribuer à la réponse à la problématique avec *select_colonne* et la variable *colonnes_a_garder* (= liste contenant toutes les noms des colonnes à garder).

De même, *rename_colonne* et *nouveaux_noms* nous permettent de renommer :

In [8]:
df_ess_select = select_colonne(df_ess_2020, colonnes_a_garder)
df_ess_rename = renommer_colonne(df_ess_select, colonnes_a_garder, nouveaux_noms)
print(df_ess_rename.shape)
print(df_ess_rename.columns)

(37611, 39)
Index(['id', 'pays', 'poids_analyse', 'poids_design', 'poids_poststrat',
       'complot_groupes_secrets', 'complot_science_politique',
       'dissimulation_covid', 'mesures_sanitaires_controle',
       'mesures_sanitaires_equilibre', 'confiance_science', 'refus_vaccin',
       'statut_vaccine', 'axe_gauche_droite', 'interet_politique', 'a_vote',
       'proche_parti', 'degre_proximite_parti', 'confiance_parlement',
       'confiance_politiques', 'confiance_partis', 'confiance_justice',
       'confiance_police', 'efficacite_politique_citoyen',
       'prise_en_compte_citoyen', 'niveau_etudes_isced', 'annees_etudes',
       'decile_revenu', 'sentiment_revenu', 'activite_principale', 'chomage',
       'milieu_residence', 'appartenance_religieuse', 'degre_religiosite',
       'frequence_culte', 'frequence_priere', 'genre', 'age',
       'annee_naissance'],
      dtype='str')


### 2. Valeurs manquantes

Regardons les variables ayant +5% de valeurs manquantes.

In [9]:
df_val_manquant = pd.DataFrame({
    "Nbr valeurs manquantes": df_ess_rename.isnull().sum(),
    "% valeurs manquantes": df_ess_rename.isnull().mean() * 100
})

df_val_manquant = df_val_manquant[df_val_manquant["% valeurs manquantes"] >= 5]
df_val_manquant = df_val_manquant.sort_values(by="% valeurs manquantes", ascending=False)

print(df_val_manquant)

                              Nbr valeurs manquantes  % valeurs manquantes
refus_vaccin                                   37073             98.569567
activite_principale                            33940             90.239558
degre_proximite_parti                          23289             61.920715
statut_vaccine                                 20679             54.981255
decile_revenu                                   8239             21.905825
confiance_science                               7866             20.914094
dissimulation_covid                             5762             15.319986
complot_groupes_secrets                         5632             14.974343
axe_gauche_droite                               5297             14.083646
complot_science_politique                       5293             14.073011
mesures_sanitaires_equilibre                    5094             13.543910
mesures_sanitaires_controle                     3928             10.443753


On supprime les variables refus_vaccin, activite_principale, degre_proximite_parti, statut_vaccine qui ont une part trop élevée de valeurs manquantes :

In [10]:
df_ess_suppr = suppr_colonne(df_ess_rename, 
    ["refus_vaccin", "activite_principale", "degre_proximite_parti", "statut_vaccine"])

print(df_ess_suppr.shape)

(37611, 35)


Quant aux autres variables ayant entre 10 % et 20 % de valeurs sont manquantes, on fait le choix de ne pas les supprimer. Lors de l'analyse descriptive, les valeurs manquantes seront exclues du calcul et regroupées dans une catégorie "Non renseigné". 

En consultant la documentation méthodologique officielle de l'ESS Round 10 (*voir fichier README.md*), on remarque une absence totale de certaines valeurs dans certains pays, s'expliquant par les contraintes imposées par la crise sanitaire sur les modes de collecte.


<details style="border: 1px solid #888; border-radius: 6px; padding: 8px;">
<summary style="font-weight: bold; cursor: pointer;">Explication détaillée des valeurs manquantes à l'aide de la documentation</summary>

1. **Raccourcissement du questionnaire pour l'auto-passation (Self-Completion) :**  
   Plusieurs pays, dont le Monténégro (ME) et le Royaume-Uni (GB), ont dû abandonner le traditionnel entretien en face-à-face pour basculer en urgence vers un mode d'auto-administration (questionnaire web et papier par courrier). Pour limiter le fardeau cognitif et les taux d'abandon inhérents aux questionnaires sans enquêteur, l'ESS a drastiquement allégé le questionnaire, supprimant certains modules rotatifs secondaires dont les questions spécifiques sur la science (`trstsci`) ou les décisions secrètes.

2. **Divergences nationales d'implémentation du module Covid-19 :**  
   Pour la France (FR), le terrain d'enquête a été décalé et restructuré sous le format face-à-face assisté par ordinateur (CAPI), mais avec des arbitrages temporels stricts sur la durée de passation. L'équipe nationale de coordination a priorisé le tronc commun et le module démocratie, omettant l'extension facultative consacrée à la perception de la science et aux attitudes sanitaires alternatives.

Ces valeurs manquantes ne relèvent donc pas d'une non-réponse individuelle aléatoire (*Missing at Random*), mais d'un choix institutionnel de conception de l'enquête (*Structural Missings* / *Not Available*). Pour notre modélisation, cela implique de restreindre les analyses comparatives aux pays ayant administré l'intégralité du module (ou d'introduire des effets fixes pays pour neutraliser ce biais structurel).

</details>

Ceci nous invite donc à regarder quels sont les pays où il y a beaucoup de valeurs manquantes pour ces raisons et à les supprimer :

In [12]:
df_par_pays = df_ess_suppr.groupby("pays").agg(lambda col: col.isnull().mean() * 100)

colonne_ayant_bcp_val_manq = ["decile_revenu",
    "confiance_science",
    "dissimulation_covid",
    "complot_groupes_secrets",
    "axe_gauche_droite",
    "complot_science_politique",
    "mesures_sanitaires_equilibre",
    "mesures_sanitaires_controle"]

# On garde les variables ayant +5% de valeurs manquantes
df_par_pays = df_par_pays[colonne_ayant_bcp_val_manq]

df_par_pays_decile_revenu = df_par_pays.sort_values(
    by=["decile_revenu"],
    ascending=False
)

print(df_par_pays_decile_revenu)


      decile_revenu  confiance_science  dissimulation_covid  \
pays                                                          
GR        46.087889           0.357270             7.002501   
IT        38.371212           0.946970             7.765152   
PT        34.276387           5.277476            23.014146   
IE        32.994350           3.446328            11.073446   
HR        30.527638           2.198492             6.469849   
CZ        30.452342         100.000000             6.744750   
SK        29.971791           2.045134            11.424542   
HU        26.933478           2.217415            16.711736   
MK        23.582925           3.778866             7.837649   
CH        17.662508           3.348654             7.025607   
LT        16.937914           2.772755             6.992164   
BG        16.629875           1.545254             6.144224   
GB        12.445605           1.044386             1.740644   
BE        12.229679           0.372856             2.83

L'examen détaillé des données révèle que le volume élevé de valeurs manquantes n'est pas homogène : il résulte principalement d'une absence totale de passation de ces variables dans certains pays partenaires (notamment 100 % de valeurs manquantes pour `confiance_science` en Estonie, Tchéquie, Monténégro et France, et 100 % pour `dissimulation_covid` au Monténégro et en France). 




Afin de préserver la validité de nos indicateurs synthétiques et de garantir la comparabilité des résultats, il apparaît donc préférable d'exclure ces pays (en particulier la France et le Monténégro) du champ de notre étude principale, quitte à réduire légèrement la taille de l'échantillon global.


In [64]:
df_ess_suppr_pays = df_ess_suppr[~df_ess_suppr["pays"].isin(["FR", "ME"])]

print(df_ess_suppr_pays.shape)

(34356, 35)


Regardons à nouveau si la part des valeurs manquantes a diminué :

In [65]:
df_val_manquant_2 = pd.DataFrame({
    "Nbr valeurs manquantes": df_ess_suppr_pays.isnull().sum(),
    "% valeurs manquantes": df_ess_suppr_pays.isnull().mean() * 100
})

df_val_manquant_2 = df_val_manquant_2.sort_values(by="% valeurs manquantes", ascending=False)

print(df_val_manquant_2.head(10))

                              Nbr valeurs manquantes  % valeurs manquantes
decile_revenu                                   7888             22.959599
axe_gauche_droite                               4662             13.569682
confiance_science                               4611             13.421236
dissimulation_covid                             2507              7.297124
complot_groupes_secrets                         2377              6.918733
complot_science_politique                       2038              5.932006
mesures_sanitaires_equilibre                    1839              5.352777
proche_parti                                     825              2.401327
efficacite_politique_citoyen                     794              2.311096
mesures_sanitaires_controle                      673              1.958901


Faire un commentaire sur les résultats (est ce que la part des valeurs manquantes a bien baissé ? )

Remplacer valeurs manquante par mean ou median ou les drop ? Regarder les types de variables. Normaliser les variables ? Affiche la taille, decrire dataframe final 

PS : g pas fini

### 3. Création de variables continues/agrégées


Afin d'opérationnaliser notre problématique sans surcharger la modélisation, nous avons fait le choix de réduire drastiquement le nombre de dimensions en créant des indicateurs synthétiques. Au vu de la structure du jeu de données, nous restreignons notre analyse à un ensemble final de 10 variables clés : 3 variables expliquées et 7 variables explicatives (mêlant indicateurs construits et variables brutes directement pertinentes).

1. **Variables expliquées (mesure de l'adhésion aux complots) :**
   * `score_complot` *(créée)* : indicateur continu (de 0 à 10) mesurant le niveau moyen d'adhésion aux récits complotistes (moyenne synthétique des questions sur les groupes secrets, la collusion politico-scientifique et le Covid-19).
   * `adhesion_complot_forte` *(créée)* : indicatrice binaire (0 ou 1) identifiant les profils complotistes affirmés (accord $\ge 4/5$ sur au moins deux items), adaptée aux futures régressions logistiques.
   * `dissimulation_covid` *(brute)* : variable ordinale mesurant la perception de rétention d'informations par les autorités lors de la crise sanitaire.

2. **Variables explicatives (déterminants politiques, culturels et structurels) :**
   * `confiance_institutions` *(créée)* : score continu (0 à 10) agrégeant la confiance dans le parlement, le personnel politique, les partis et la justice.
   * `score_socioeco` *(créée)* : indicateur composite continu (borné de 0 à 1, inspiré de l'IDH) combinant le décile de revenu du ménage et le nombre d'années d'études.
   * `radicalite_politique` *(créée)* : distance absolue au centre (5) sur l'axe gauche-droite pour capter l'effet des extrêmes.
   * `zone_rurale` *(créée)* : indicatrice binaire distinguant l'habitat rural/campagne du milieu urbain.
   * `confiance_science` *(brute)* : échelle continue (0 à 10) mesurant la confiance envers le consensus scientifique.
   * `degre_religiosite` *(brute)* : échelle continue (0 à 10) mesurant l'intensité de la foi personnelle.
   * `age` *(brute)* : âge exact du répondant pour contrôler les effets de cohorte.

Ce resserrement méthodologique permet de disposer d'indicateurs continus robustes pour les régressions linéaires tout en évitant les problèmes de multicolinéarité liés à la multiplication de questions similaires.



In [ ]:
# 1) nettoyage préventif des codes de non-réponse
complot_cols = ["complot_groupes_secrets", "complot_science_politique", "dissimulation_covid"]
for col in complot_cols:
    df_clean.loc[~df_clean[col].between(1, 5), col] = np.nan

inst_cols = ["confiance_parlement", "confiance_politiques", "confiance_partis", "confiance_justice"]
for col in inst_cols:
    df_clean.loc[~df_clean[col].between(0, 10), col] = np.nan

df_clean.loc[~df_clean["axe_gauche_droite"].between(0, 10), "axe_gauche_droite"] = np.nan
df_clean.loc[~df_clean["decile_revenu"].between(1, 10), "decile_revenu"] = np.nan
df_clean.loc[~df_clean["annees_etudes"].between(0, 25), "annees_etudes"] = np.nan
df_clean.loc[~df_clean["degre_religiosite"].between(0, 10), "degre_religiosite"] = np.nan
df_clean.loc[~df_clean["confiance_science"].between(0, 10), "confiance_science"] = np.nan


# 2) construction des variables expliquées (Y)
    # A) score synthétique continu de complotisme ramené sur une échelle de 0 à 10
df_clean["score_complot"] = (df_clean[complot_cols].mean(axis=1) - 1) * 2.5

    # B) indicatrice d'adhésion forte (binaire : 1 si accord fort sur >= 2 items, 0 sinon)
df_clean["adhesion_complot_forte"] = ((df_clean[complot_cols] >= 4).sum(axis=1) >= 2).astype(int)


# 3) construction des variables explicatives synthétiques (X)
    # C) confiance institutionnelle moyenne (continue, 0 à 10)
df_clean["confiance_institutions"] = df_clean[inst_cols].mean(axis=1)

    # D) score socio-économique composite normalisé (continu de 0 à 1)
rev_norm = (df_clean["decile_revenu"] - 1) / 9.0
edu_norm = df_clean["annees_etudes"] / 25.0
df_clean["score_socioeco"] = (rev_norm + edu_norm) / 2.0

    # E) radicalité politique (distance au centre 5 sur l'axe gauche-droite)
df_clean["radicalite_politique"] = (df_clean["axe_gauche_droite"] - 5).abs()

    # F) zone rurale (binaire : 1 si campagne/village, 0 si ville)
df_clean["zone_rurale"] = df_clean["milieu_residence"].isin([4, 5]).astype(int)


# 4) définition des listes de variables pour les régressions et visualisations

variables_expliquees = [
    "score_complot", 
    "adhesion_complot_forte", 
    "dissimulation_covid"
]

variables_explicatives = [
    "confiance_institutions",
    "confiance_science",
    "radicalite_politique",
    "score_socioeco",
    "degre_religiosite",
    "zone_rurale",
    "age"
]

variables_finales = variables_expliquees + variables_explicatives
df_clean[variables_finales].head()


## III. Visualisation 

Cette partie consiste en l'analyse et la visualsation...............

### 1. Statistiques descriptives

In [ ]:
# faire dataframe ou graphiques comme dans la 6 du fichier stats desc qui monte les stats comme moyenne, etc. 
# des différentes variables

Faire des "commentaires dessus"

### 2. Analyse descriptive

Graphiques basique (histogramme, boxplot qui permet situer)

Graphiques détaillant relation entre les variables (genre confiance_science en fonction du decile revenu etc..)

## IV. Conclusion

Ceci est une conclusion